In [1]:
import torch
import transformers
import modelscope

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("modelscope:", modelscope.__version__)
print("环境测试成功")

/mnt/workspace/llm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.12.0+cpu
transformers: 5.9.0
modelscope: 1.37.1
环境测试成功


In [2]:
from modelscope.hub.snapshot_download import snapshot_download

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

model_dir = snapshot_download(
    model_id,
    cache_dir="/mnt/workspace/models"
)

print("模型下载完成：", model_dir)

2026-05-26 10:58:56,387 - modelscope - INFO - Got 10 files, start to download ...
Processing 10 items:   0%|          | 0.00/10.0 [00:00<?, ?it/s]































Processing 10 items:  10%|█         | 1.00/10.0 [00:00<00:04, 1.95it/s]














Processing 10 items:  40%|████      | 4.00/10.0 [00:00<00:00, 7.73it/s]































Processing 10 items:  70%|███████   | 7.00/10.0 [00:00<00:00, 9.71it/s]19, 50.4MB/s]












Processing 10 items:  90%|█████████ | 9.00/10.0 [00:01<00:00, 8.57it/s]








































































































































































































Processing 10 items: 100%|██████████| 10.0/10.0 [00:05<00:00, 1.67it/s]
2026-05-26 10:59:02,388 - modelscope - INFO - Finish downloading 10 files for repo 'Qwen/Qwen2.5-0.5B-Instruct'
2026-05-26 10:59:02,390 - modelscope - INFO - Creating symbolic link [/mnt/works

模型下载完成： /mnt/workspace/models/Qwen/Qwen2___5-0___5B-Instruct


In [3]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    model_dir,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True
)

model.eval()

def ask(question):
    messages = [
        {"role": "system", "content": "你是一个中文问答助手，请回答得清楚、简洁。"},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt")

    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    end = time.time()

    print("问题：", question)
    print("回答：", answer)
    print(f"耗时：{end - start:.2f} 秒")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:02<00:00, 125.02it/s]


In [4]:
ask("请说出以下两句话区别在哪里？1、冬天：能穿多少穿多少。2、夏天：能穿多少穿多少。")

问题： 请说出以下两句话区别在哪里？1、冬天：能穿多少穿多少。2、夏天：能穿多少穿多少。
回答： 这两句话的区别在于“冬天”和“夏天”的定义不同。

在“冬天”，通常指的是冬季，即从冬至开始到春分结束的这段时间。在这个时间段内，天气寒冷，户外活动受限，人们需要保暖穿着多。

而在“夏天”，则指夏季，是从夏至开始到秋分结束的这段时间。在这个时间段内，天气炎热，户外活动不受限制，人们可以享受更多的阳光和自然环境。

所以，“冬天”和“夏天”的定义是不同的，它们对应的是不同的季节。
耗时：5.84 秒


In [5]:
ask("请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上。")

问题： 请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上。
回答： 这两句话的区别在于：

1. 第一句提到的是“单身狗”，而第二句则提到了“单身狗”和“单身汉”。

2. 第一句中，“谁”指的是所有人，而第二句中的“谁”是指特定的人。

3. 第一句中，“都”表示所有人的意思，而第二句中的“都”表示所有人都的意思。

4. 第一句中，“都”表示所有人的意思，而第二句中的“都”表示所有人都的意思。

5. 第一句中，“都”表示所有人的意思，而第二句中的“都”表示所有人都的意思。

6. 第一句中，“都”表示所有人的意思，而第二句中的“都”表示所有人都的意思。

7. 第一句中，“都”表示所有人的意思，而第二句中的“都”表示所有人都的意思。

8. 第一句中，“都”表示所有人的意思，而第二句中的“都”表示所有人都的意思。

9. 第一句
耗时：10.15 秒


In [6]:
ask("他知道我知道你知道他不知道吗？这句话里，到底谁不知道？")

问题： 他知道我知道你知道他不知道吗？这句话里，到底谁不知道？
回答： 这句话里的“他”指的是“他不知道”。
耗时：0.73 秒


In [7]:
ask("明明明明明白白白喜欢他，可她就是不说。这句话里，明明和白白谁喜欢谁？")

问题： 明明明明明白白白喜欢他，可她就是不说。这句话里，明明和白白谁喜欢谁？
回答： 这句话里的“明明”是说白某人喜欢白某人，“白白”是说白某人喜欢白某人。所以，明明喜欢白白。
耗时：1.84 秒


In [8]:
import gc

del model
del tokenizer
gc.collect()

6

In [9]:
from modelscope.hub.snapshot_download import snapshot_download

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

model_dir = snapshot_download(
    model_id,
    cache_dir="/mnt/workspace/models"
)

print("模型下载完成：", model_dir)

2026-05-26 11:33:56,787 - modelscope - INFO - Got 10 files, start to download ...
Processing 10 items:   0%|          | 0.00/10.0 [00:00<?, ?it/s]







































Processing 10 items:  10%|█         | 1.00/10.0 [00:00<00:04, 1.84it/s]







































Processing 10 items:  80%|████████  | 8.00/10.0 [00:00<00:00, 10.3it/s]:40, 19.1MB/s]











































































































































































































































































































































































































































































































































































































































模型下载完成： /mnt/workspace/models/Qwen/Qwen2___5-1___5B-Instruct


In [10]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    model_dir,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True
)

model.eval()

def ask(question):
    messages = [
        {"role": "system", "content": "你是一个中文问答助手，请回答得清楚、简洁。"},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt")

    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    end = time.time()

    print("问题：", question)
    print("回答：", answer)
    print(f"耗时：{end - start:.2f} 秒")

Loading weights: 100%|██████████| 338/338 [00:06<00:00, 54.16it/s]


In [11]:
ask("请说出以下两句话区别在哪里？1、冬天：能穿多少穿多少。2、夏天：能穿多少穿多少。")

问题： 请说出以下两句话区别在哪里？1、冬天：能穿多少穿多少。2、夏天：能穿多少穿多少。
回答： 这句话的意思是，冬天可以穿很多衣服，夏天也可以穿很多衣服。
耗时：2.41 秒


In [12]:
ask("请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上。")

问题： 请说出以下两句话区别在哪里？单身狗产生的原因有两个，一是谁都看不上，二是谁都看不上。
回答： 第一个句子是错的，第二个句子是对的。
耗时：1.79 秒


In [13]:
ask("他知道我知道你知道他不知道吗？这句话里，到底谁不知道？")

问题： 他知道我知道你知道他不知道吗？这句话里，到底谁不知道？
回答： 这句话的逻辑混乱，无法确定具体哪个人不知道。
耗时：1.89 秒


In [14]:
ask("明明明明明白白白喜欢他，可她就是不说。这句话里，明明和白白谁喜欢谁？")

问题： 明明明明明白白白喜欢他，可她就是不说。这句话里，明明和白白谁喜欢谁？
回答： 明明喜欢白白。
耗时：0.98 秒
